# Model Iteration Notebook 

In this notebook, we will focus on training and evaluating multiple machine learning models to classify emotions in transcribed television show data. 

The different secitons in this notebook will help demonstrate how different models perform on the same dataset and will provide insights into model selection and optimization. 

For every type of model and/or iteration that we did, we tracked the details in the model iteration file that you can find attached to the final assignment on Brightspace or here [Model Iteration File](link-here). In order to improve our models, we used (a selection of) traditional NLP features that we extracted in the ‘NLP Features’ task [NLP Features.ipynb notebook](NLP Features.ipynb). 



In [1]:
import sys
import os

# Get the directory of the current notebook (which is /notebooks/)
# and navigate up one level (to /project_root/) to find /src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to the system path
if project_root not in sys.path:
    sys.path.append(project_root)

# Now you can import as if you were in the project root
# For /src/processing/features.py, the module is src.processing.features
from src.processing.features import FeatureEngine

c:\Users\rzvn1\.conda\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\rzvn1\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rzvn1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rzvn1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\rzvn1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\rzvn1

In [2]:
import numpy as np
import pandas as pd
from typing import Dict

In [3]:
import torch 

print("Torch version:",torch.__version__)

print("Is CUDA enabled?",torch.cuda.is_available())

Torch version: 2.5.1
Is CUDA enabled? True


In [4]:
# Emotion Map obtained from https://aclanthology.org/I17-1099/ --> readme.txt
EMOTION_MAP: Dict[int, str] = {
    0: "neutral", 1: "anger", 2: "disgust", 3: "fear",
    4: "happiness", 5: "sadness", 6: "surprise"
}

In [5]:
df_concat = pd.read_csv(r"..\Data\CSV\sentiment_data\final_dataset.csv")

### Data preparation for scikit-learn

In [6]:
scikit_data = []
# Get unique emotions
unique_emotions = df_concat['Emotion'].unique()

# For each unique emotion, add 350 different rows
for emotion in unique_emotions:
    # Get all rows with this emotion
    emotion_rows = df_concat[df_concat['Emotion'] == emotion]
    
    # Sample 350 rows (with replacement if there are fewer than 350 available)
    sampled_rows = emotion_rows.sample(n=350, replace=True, random_state=42)
    
    # Add to the list
    scikit_data.append(sampled_rows)

# Create new dataframe by concatenating all sampled rows
scikit_data = pd.concat(scikit_data, ignore_index=True)

scikit_data['Emotion_ID'] = scikit_data['Emotion_ID'].astype(int)

scikit_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2450 entries, 0 to 2449
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  2450 non-null   int64 
 1   Sentence    2450 non-null   object
 2   Emotion     2450 non-null   object
 3   Emotion_ID  2450 non-null   int32 
dtypes: int32(1), int64(1), object(2)
memory usage: 67.1+ KB


In [7]:
scikit_data['Sentence'].nunique()
scikit_data.drop_duplicates(subset=['Sentence'], inplace=True)

In [8]:
features = FeatureEngine()

results = features.create_features(transcript_df_input=scikit_data, output_path=None)

Training Word2Vec model...
Training complete.


In [9]:
scikit_df = results.copy()

In [10]:
results.head()

,Unnamed: 0,Sentence,Emotion,Emotion_ID,POS_tags,Sentiment,TF-IDF,word2vec_embedding,custom_word2vec_embedding,bert_embedding
0,131607,i think the fact that i am feeling a little ap...,fear,3,"[(i, NOUN), (think, VERB), (the, DET), (fact, ...",0.137500,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.0067983773, 0.028879605, 0.0091975285, 0.0...","[0.091844276, 0.2581376, -0.083199434, 0.08026...","[0.26444843, 0.39232564, 0.06754342, -0.534239..."
1,6835,i started feeling terrified,fear,3,"[(i, NOUN), (started, VERB), (feeling, VERB), ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.030395508, 0.08758545, 0.0077209473, 0.0012...","[0.20521021, 0.44118753, -0.05267038, -0.06366...","[0.12794185, 0.13075173, 0.14813915, -0.064277..."
2,330814,i feel for all of the victims and their famili...,fear,3,"[(i, NOUN), (feel, VERB), (for, ADP), (all, DE...",0.166667,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.010828191, 0.02356512, 0.010276101, 0.12585...","[0.07549258, 0.14661953, -0.09227799, 0.106705...","[-0.06701388, 0.14782472, 0.12844124, -0.26227..."
3,506845,I’m afraid of the dark.,fear,3,"[(I, PRON), (’, VERB), (m, ADJ), (afraid, NOUN...",-0.375000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.1953125, 0.08569336, 0.13183594, 0.0579834,...","[0.14037292, 0.062451404, -0.11458135, 0.35504...","[0.30258685, 0.33086362, -0.35033846, -0.45129..."
4,93206,i feel pretty shy about the students but i thi...,fear,3,"[(i, NOUN), (feel, VERB), (pretty, ADV), (shy,...",0.170833,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.029205322, 0.029956818, 0.04335022, 0.12544...","[0.04682644, 0.1441369, -0.079293266, 0.120165...","[0.28519934, 0.10159663, 0.25914514, -0.241023..."


In [11]:
scikit_df = scikit_df.drop(columns=['Emotion', 'POS_tags','TF-IDF','custom_word2vec_embedding', 'Sentence'])

In [12]:
scikit_df['Sentiment'] = scikit_df['Sentiment'].values.reshape(-1, 1)
scikit_df['word2vec_embedding'] = np.vstack(scikit_df['word2vec_embedding'].values)
scikit_df['bert_embedding'] = np.vstack(scikit_df['bert_embedding'].values)

### Logistic Regression

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

log_reg = LogisticRegression(max_iter=1000, random_state=42)
X = scikit_df.drop(columns=['Emotion_ID'])
y = scikit_df['Emotion_ID']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)

print(classification_report(y_test, y_pred, target_names=EMOTION_MAP.values()))


              precision    recall  f1-score   support

     neutral       0.77      0.80      0.78        75
       anger       0.22      0.25      0.23        65
     disgust       0.55      0.88      0.68        64
        fear       0.30      0.37      0.33        70
   happiness       0.48      0.56      0.51        70
     sadness       0.50      0.04      0.07        83
    surprise       0.12      0.11      0.12        61

    accuracy                           0.42       488
   macro avg       0.42      0.43      0.39       488
weighted avg       0.43      0.42      0.39       488



In [14]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB(var_smoothing=0.0005)
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print(classification_report(y_test, y_pred_nb, target_names=EMOTION_MAP.values()))

              precision    recall  f1-score   support

     neutral       0.79      0.91      0.84        75
       anger       0.00      0.00      0.00        65
     disgust       0.55      0.95      0.70        64
        fear       0.23      0.46      0.30        70
   happiness       0.30      0.49      0.37        70
     sadness       0.00      0.00      0.00        83
    surprise       0.17      0.10      0.12        61

    accuracy                           0.41       488
   macro avg       0.29      0.41      0.33       488
weighted avg       0.29      0.41      0.33       488



c:\Users\rzvn1\.conda\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\rzvn1\.conda\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\rzvn1\.conda\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0]

In [15]:
from sklearn.svm import LinearSVC

svc = LinearSVC(penalty='l2', loss='squared_hinge', C=500.0, multi_class='ovr', fit_intercept=True, random_state=42)
svc.fit(X_train, y_train)

y_pred_svc = svc.predict(X_test)
print(classification_report(y_test, y_pred_svc, target_names=EMOTION_MAP.values()))

              precision    recall  f1-score   support

     neutral       0.00      0.00      0.00        75
       anger       0.00      0.00      0.00        65
     disgust       0.13      1.00      0.23        64
        fear       0.00      0.00      0.00        70
   happiness       0.00      0.00      0.00        70
     sadness       0.00      0.00      0.00        83
    surprise       0.00      0.00      0.00        61

    accuracy                           0.13       488
   macro avg       0.02      0.14      0.03       488
weighted avg       0.02      0.13      0.03       488



c:\Users\rzvn1\.conda\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\rzvn1\.conda\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\rzvn1\.conda\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0]

In [16]:
# Set Keras Backend to use PyTorch
os.environ["KERAS_BACKEND"] = "torch"
import keras

In [17]:
keras_data = []
# Get unique emotions
unique_emotions = df_concat['Emotion'].unique()

# For each unique emotion, add 350 different rows
for emotion in unique_emotions:
    # Get all rows with this emotion
    emotion_rows = df_concat[df_concat['Emotion'] == emotion]
    
    # Sample 350 rows (with replacement if there are fewer than 350 available)
    sampled_rows = emotion_rows.sample(n=1000, replace=True, random_state=42)
    
    # Add to the list
    keras_data.append(sampled_rows)

# Create new dataframe by concatenating all sampled rows
keras_data = pd.concat(keras_data, ignore_index=True)
keras_data['Emotion_ID'] = keras_data['Emotion_ID'].astype(int)

keras_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  7000 non-null   int64 
 1   Sentence    7000 non-null   object
 2   Emotion     7000 non-null   object
 3   Emotion_ID  7000 non-null   int32 
dtypes: int32(1), int64(1), object(2)
memory usage: 191.5+ KB


In [18]:
keras_data['Sentence'].nunique()
keras_data.drop_duplicates(subset=['Sentence'], inplace=True)

In [19]:
features = FeatureEngine()

keras_df = features.create_features(transcript_df_input=keras_data, output_path=None)

Training Word2Vec model...
Training complete.


In [20]:
keras_df.drop(columns='Unnamed: 0', inplace=True)

In [21]:
keras_df.head()

,Sentence,Emotion,Emotion_ID,POS_tags,Sentiment,TF-IDF,word2vec_embedding,custom_word2vec_embedding,bert_embedding
0,i think the fact that i am feeling a little ap...,fear,3,"[(i, NOUN), (think, VERB), (the, DET), (fact, ...",0.137500,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.0067983773, 0.028879605, 0.0091975285, 0.0...","[-0.087505706, 0.17178163, 0.01258083, 0.11640...","[0.26444843, 0.39232564, 0.06754342, -0.534239..."
1,i started feeling terrified,fear,3,"[(i, NOUN), (started, VERB), (feeling, VERB), ...",0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.030395508, 0.08758545, 0.0077209473, 0.0012...","[-0.08090373, 0.25339434, 0.08808849, -0.01455...","[0.12794185, 0.13075173, 0.14813915, -0.064277..."
2,i feel for all of the victims and their famili...,fear,3,"[(i, NOUN), (feel, VERB), (for, ADP), (all, DE...",0.166667,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.010828191, 0.02356512, 0.010276101, 0.12585...","[-0.06610727, 0.15845835, 0.04844751, 0.098227...","[-0.06701388, 0.14782472, 0.12844124, -0.26227..."
3,I’m afraid of the dark.,fear,3,"[(I, PRON), (’, VERB), (m, ADJ), (afraid, NOUN...",-0.375000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.1953125, 0.08569336, 0.13183594, 0.0579834,...","[-0.1696652, -0.078601874, 0.17643587, 0.33382...","[0.30258685, 0.33086362, -0.35033846, -0.45129..."
4,i feel pretty shy about the students but i thi...,fear,3,"[(i, NOUN), (feel, VERB), (pretty, ADV), (shy,...",0.170833,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.029205322, 0.029956818, 0.04335022, 0.12544...","[-0.14402393, 0.13064769, 0.02586022, 0.128485...","[0.28519934, 0.10159663, 0.25914514, -0.241023..."


In [22]:
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [23]:
keras_df = keras_df.drop(columns=['Emotion', 'POS_tags','TF-IDF','bert_embedding'])

In [24]:
X = keras_df.drop(columns=['Emotion_ID'])
y = keras_df['Emotion_ID']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
X_train.shape, X_test.shape

((5498, 4), (1375, 4))

In [26]:
tokenizer = Tokenizer(filters='')
tokenizer.fit_on_texts(X_train['Sentence'])

In [27]:
V = len(tokenizer.word_index)
V

9924

In [28]:
train_tok = tokenizer.texts_to_sequences(X_train['Sentence'])
test_tok = tokenizer.texts_to_sequences(X_test['Sentence'])

In [29]:
train_padded = pad_sequences(train_tok,
              padding='post',
              maxlen=30)

test_padded = pad_sequences(test_tok,
              padding='post',
              maxlen=30)


In [30]:
X_train['Sentence_Tok'] = train_padded.tolist()
X_test['Sentence_Tok'] = test_padded.tolist()

In [31]:
X_train.drop(columns='Sentence', inplace=True)
X_test.drop(columns='Sentence', inplace=True)

In [32]:
from keras.utils import to_categorical
y_train = to_categorical(y_train, num_classes=7)
y_test = to_categorical(y_test, num_classes=7)

In [33]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((5498, 4), (5498, 7), (1375, 4), (1375, 7))

In [34]:
X_train.head()

,Sentiment,word2vec_embedding,custom_word2vec_embedding,Sentence_Tok
4688,0.20,"[0.029514018, 0.015031081, 0.016991835, 0.1229...","[-0.12079874, 0.15417543, 0.0018635994, 0.0655...","[1, 5, 18, 2, 60, 1, 19, 24, 2, 677, 15, 63, 4..."
5693,0.00,"[0.07672119, -0.006591797, 0.04321289, 0.08046...","[-0.14353804, 0.15102655, 0.06547556, 0.103316...","[128, 25, 1542, 4162, 170, 2, 884, 40, 0, 0, 0..."
2134,0.00,"[0.04073819, 0.02465959, 0.081809305, 0.135409...","[-0.025940618, 0.12760887, 1.1826307e-05, 0.08...","[1, 33, 4163, 32, 2807, 1, 252, 75, 2808, 973,..."
252,0.00,"[0.012212117, 0.040827435, -0.017425537, 0.117...","[-0.17353362, 0.19123429, -0.08085985, 0.07847...","[1, 27, 90, 4165, 4, 4166, 313, 5, 1, 27, 98, ..."
2666,0.35,"[0.033184346, -0.00034038836, 0.024799053, 0.1...","[-0.18975355, 0.062441785, 0.016794631, 0.1450...","[1, 5, 63, 1090, 3, 30, 322, 3, 4167, 4168, 13..."


In [35]:
X_train = np.hstack([
    np.stack(X_train["word2vec_embedding"].to_numpy()),
    np.stack(X_train["custom_word2vec_embedding"].to_numpy()),
    X_train["Sentiment"].to_numpy().reshape(-1, 1)  # keep sentiment too
])

X_test = np.hstack([
    np.stack(X_test["word2vec_embedding"].to_numpy()),
    np.stack(X_test["custom_word2vec_embedding"].to_numpy()),
    X_test["Sentiment"].to_numpy().reshape(-1, 1)
])

In [79]:
from keras import Sequential
from keras.layers import (Embedding, Dense, 
                          SimpleRNN, LSTM)
from keras.callbacks import EarlyStopping
from keras.optimizers import AdamW

In [80]:
model = Sequential([
        # Recurrent Layer
        # SimpleRNN(128, activation='tanh'),

        # Dense Layers
        Dense(128, activation='relu'),
        Dense(128, activation='relu'),

        # Output Layer
        Dense(7, activation='softmax')
    ])

In [81]:
model.compile(optimizer=AdamW(learning_rate=5e-4), loss='categorical_crossentropy', metrics=['accuracy'])

In [82]:
# Monitors the validation loss and stops after 3 epochs with no improvement
early_stopping = EarlyStopping(monitor='val_loss', patience=3, verbose=0, restore_best_weights=True)

# Fit the model to the training data.
model.fit(
    X_train,
    y_train,
    epochs=30,
    verbose=1,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping] 
)

Epoch 1/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3592 - loss: 1.6743 - val_accuracy: 0.4276 - val_loss: 1.4250
Epoch 2/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5104 - loss: 1.3160 - val_accuracy: 0.5607 - val_loss: 1.2081
Epoch 3/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5926 - loss: 1.1428 - val_accuracy: 0.6233 - val_loss: 1.0724
Epoch 4/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6430 - loss: 1.0297 - val_accuracy: 0.6444 - val_loss: 1.0246
Epoch 5/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6744 - loss: 0.9422 - val_accuracy: 0.6633 - val_loss: 0.9642
Epoch 6/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6966 - loss: 0.8889 - val_accuracy: 0.6618 - val_loss: 0.9513
Epoch 7/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7186 - loss: 0.8215 - val_accuracy: 0.6713 - val_loss: 0.9051
Epoch 8/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7352 - loss: 0.7773 - val_accuracy: 0.

In [83]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test Loss: 0.8687
Test Accuracy: 0.6916


In [84]:
# Step 1: Get predictions
y_pred_probs = model.predict(X_test)

# Step 2: Convert probabilities to class labels
y_pred = np.argmax(y_pred_probs, axis=1)

# Step 3: Convert y_test if it's one-hot encoded
if y_test.ndim > 1 and y_test.shape[1] > 1:
    y_true = np.argmax(y_test, axis=1)
else:
    y_true = y_test

# Step 4: Generate classification report
print(classification_report(y_true, y_pred, digits=2))

43/43 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
              precision    recall  f1-score   support

           0       0.77      0.69      0.73       208
           1       0.61      0.62      0.62       186
           2       0.63      0.77      0.69       174
           3       0.70      0.70      0.70       203
           4       0.73      0.70      0.72       200
           5       0.62      0.56      0.59       213
           6       0.79      0.81      0.80       191

    accuracy                           0.69      1375
   macro avg       0.69      0.69      0.69      1375
weighted avg       0.69      0.69      0.69      1375

